In [16]:
# Executar análise via script externo
import subprocess
import os

os.chdir('/home/Projetos/ProjetosDocker/data')
result = subprocess.run(['/home/Projetos/ProjetosDocker/data/.venv/bin/python', 'analysis_duckdb.py'], 
                       capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Erros:", result.stderr)

Lendo: /home/Projetos/ProjetosDocker/data/data-1772046577622.csv

✓ Total de registros: 451

Colunas da tabela:
  - id: VARCHAR
  - run_id: VARCHAR
  - status: VARCHAR
  - source: VARCHAR
  - payable_id: VARCHAR
  - supplier: VARCHAR
  - document: VARCHAR
  - history: VARCHAR
  - due_date: VARCHAR
  - amount: DOUBLE
  - movement_id: VARCHAR
  - movement_date: VARCHAR
  - movementType: VARCHAR
  - movement_desc: VARCHAR
  - movement_balance: VARCHAR
  - matched_movement_id: VARCHAR
  - created_at: TIMESTAMP
  - match_reason: VARCHAR
  - match_score: BIGINT

Distribuição de STATUS:
  MATCHED: 332 registros
  UNMATCHED: 91 registros
  UNMATCHED_MOVEMENT: 28 registros

Valor total por STATUS:
  MATCHED: R$ 798,689.54 (média: R$ 2,405.69)
  UNMATCHED: R$ 150,972.65 (média: R$ 1,659.04)
  UNMATCHED_MOVEMENT: R$ 70,697.63 (média: R$ 2,524.92)

Top 10 Suppliers:
  1861: 128 registros (R$ 439,715.76)
  421: 115 registros (R$ 50,935.71)
  1720: 65 registros (R$ 95,090.67)
  NULL: 28 registros (R

In [18]:
# Gerar visualizações
print('\n\n📊 Gerando visualizações Plotly...\n')
result = subprocess.run(['/home/Projetos/ProjetosDocker/data/.venv/bin/python', 'visualize_duckdb.py'], 
                       capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Erros:", result.stderr)

# Exibir resumo dos arquivos criados
print('\n' + '='*50)
print('📂 Arquivos de visualização gerados:')
print('='*50)
files = ['dashboard.html', 'grafico_status.html', 'grafico_valor_status.html', 
         'grafico_top_suppliers.html', 'grafico_temporal.html']
for f in files:
    fpath = f'/home/Projetos/ProjetosDocker/data/{f}'
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f'✓ {f} ({size:.1f} KB)')

print('\n🌐 Abra no navegador: file:///home/Projetos/ProjetosDocker/data/dashboard.html')



📊 Gerando visualizações Plotly...

📊 Gerando visualizações...

1️⃣  Criando gráfico de distribuição de STATUS...
   ✓ Salvo em: grafico_status.html
2️⃣  Criando gráfico de valor por STATUS...
   ✓ Salvo em: grafico_valor_status.html
3️⃣  Criando gráfico de Top 10 Suppliers...
   ✓ Salvo em: grafico_top_suppliers.html
4️⃣  Criando gráfico temporal...
   ✓ Salvo em: grafico_temporal.html
5️⃣  Criando dashboard resumido...
   ✓ Salvo em: dashboard.html

✅ Todas as visualizações foram criadas!

📂 Arquivos HTML gerados:
   1. dashboard.html - Resumo geral com links para todos os gráficos
   2. grafico_status.html - Gráfico de pizza por status
   3. grafico_valor_status.html - Gráfico de barras de valor
   4. grafico_top_suppliers.html - Top 10 suppliers
   5. grafico_temporal.html - Série temporal

🌐 Abra no navegador para visualizar os gráficos!


📂 Arquivos de visualização gerados:
✓ dashboard.html (2.7 KB)
✓ grafico_status.html (4733.5 KB)
✓ grafico_valor_status.html (4734.2 KB)
✓ graf

In [19]:
# Fazer queries customizadas direto no DuckDB
import subprocess
import json

# Função helper para fazer queries
def duckdb_query(sql):
    cmd = [
        '/home/Projetos/ProjetosDocker/data/.venv/bin/python', 
        '-c', 
        f"""
import duckdb
conn = duckdb.connect('/home/Projetos/ProjetosDocker/data/analysis.duckdb', read_only=True)
result = conn.execute('{sql}').fetchall()
for row in result:
    print(row)
"""
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/home/Projetos/ProjetosDocker/data')
    return result.stdout

# Exemplo: Registros que custam mais de R$ 10.000
print('🔍 Exemplo de Query - Registros com valor > R$ 10.000:\n')
query_result = duckdb_query("SELECT id, status, amount, supplier FROM conciliacoes WHERE amount > 10000 ORDER BY amount DESC LIMIT 5")
print(query_result)

print('\n💡 Você pode executar suas próprias queries! Exemplos:')
print('  - SELECT COUNT(*) FROM conciliacoes WHERE status = "MATCHED"')
print('  - SELECT DISTINCT supplier FROM conciliacoes')
print('  - SELECT * FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000')

🔍 Exemplo de Query - Registros com valor > R$ 10.000:

('8cfd9003-edf1-4199-8ba2-a5db72869007', 'MATCHED', 67510.62, '1861')
('13b7140b-3c51-46ac-805d-c9eeb4fb1efd', 'MATCHED', 67510.62, '2383')
('8b981b12-d996-4609-ae72-05ee19d43f3c', 'MATCHED', 56530.06, '1861')
('62ca5f6d-af5d-449a-be37-e3565bf02ef7', 'MATCHED', 56530.06, '2383')
('f52e8ab9-8cf8-46af-b18f-cf55130783fd', 'MATCHED', 45039.92, '1861')


💡 Você pode executar suas próprias queries! Exemplos:
  - SELECT COUNT(*) FROM conciliacoes WHERE status = "MATCHED"
  - SELECT DISTINCT supplier FROM conciliacoes
  - SELECT * FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000


In [20]:
# Query: Registros com valor entre R$ 1.000 e R$ 5.000
print('🔍 Registros com valor entre R$ 1.000 e R$ 5.000:\n')

query = "SELECT id, status, source, amount, supplier, created_at FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000 ORDER BY amount DESC LIMIT 20"
result = duckdb_query(query)
print(result)

# Contar quantos registros estão nessa faixa
count_query = "SELECT COUNT(*) as total FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000"
count_result = duckdb_query(count_query)
print(f'\n📊 Total de registros nessa faixa: {count_result.strip()}')

🔍 Registros com valor entre R$ 1.000 e R$ 5.000:

('1e43a283-161a-4a68-b82f-77680783ad83', 'MATCHED', 'PAID', 5000.0, '1720', datetime.datetime(2026, 2, 25, 15, 14, 58, 867000))
('74611431-29f7-43c4-a161-0eba8d4522d4', 'UNMATCHED', 'OPEN', 4981.0, 'Ademar Freire de Carvalho Neto', datetime.datetime(2026, 2, 25, 15, 14, 58, 867000))
('38ac6c36-fef3-4b3f-beed-4e30035a4f43', 'MATCHED', 'PAID', 4921.46, '1861', datetime.datetime(2026, 2, 24, 16, 51, 42, 449000))
('b5086840-ce03-424b-b801-eeeef46d345e', 'MATCHED', 'PAID', 4841.64, '1861', datetime.datetime(2026, 2, 24, 16, 55, 4, 382000))
('22e47efe-6480-421b-8198-2f96fc0a6c9e', 'UNMATCHED', 'OPEN', 4756.84, 'Receita Federal do Brasil', datetime.datetime(2026, 2, 25, 15, 14, 58, 867000))
('bd28836d-1a89-462b-a174-ef543804f5d2', 'MATCHED', 'PAID', 4660.1, '1861', datetime.datetime(2026, 2, 24, 16, 55, 4, 382000))
('b2fe8eca-e580-47b5-8a68-44770c35c932', 'MATCHED', 'PAID', 4591.52, '1861', datetime.datetime(2026, 2, 24, 16, 51, 42, 449000))
(

In [28]:
# Análise detalhada da faixa R$ 1.000 - R$ 5.000
import subprocess

cmd_base = '/home/Projetos/ProjetosDocker/data/.venv/bin/python'

print('📊 Análise detalhada - Valores entre R$ 1.000 e R$ 5.000\n')

# Total de registros
result = subprocess.run([cmd_base, '-c', """
import duckdb
conn = duckdb.connect('/home/Projetos/ProjetosDocker/data/analysis.duckdb', read_only=True)
count = conn.execute('SELECT COUNT(*) FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000').fetchone()[0]
total = conn.execute('SELECT SUM(amount) FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000').fetchone()[0]
media = conn.execute('SELECT AVG(amount) FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000').fetchone()[0]
print(f'Quantidade: {count} registros')
print(f'Valor Total: R$ {total:,.2f}')
print(f'Valor Médio: R$ {media:,.2f}')
"""], capture_output=True, text=True)
print(result.stdout)

# Por status
print('\nDistribuição por Status:')
result = subprocess.run([cmd_base, '-c', """
import duckdb
conn = duckdb.connect('/home/Projetos/ProjetosDocker/data/analysis.duckdb', read_only=True)
data = conn.execute('SELECT status, COUNT(*) as qtd, SUM(amount) as total FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000 GROUP BY status ORDER BY qtd DESC').fetchall()
for row in data:
    print(f'  {row[0]:20s} - {row[1]:3d} registros (R$ {row[2]:,.2f})')
"""], capture_output=True, text=True)
print(result.stdout)

# Top 5 suppliers
print('Top 5 Suppliers:')
result = subprocess.run([cmd_base, '-c', """
import duckdb
conn = duckdb.connect('/home/Projetos/ProjetosDocker/data/analysis.duckdb', read_only=True)
data = conn.execute('SELECT supplier, COUNT(*) as qtd, SUM(amount) as total FROM conciliacoes WHERE amount BETWEEN 1000 AND 5000 AND supplier IS NOT NULL GROUP BY supplier ORDER BY qtd DESC LIMIT 5').fetchall()
for row in data:
    print(f'  {row[0][:30]:30s} - {row[1]:3d} registros (R$ {row[2]:,.2f})')
"""], capture_output=True, text=True)
print(result.stdout)

📊 Análise detalhada - Valores entre R$ 1.000 e R$ 5.000

Quantidade: 105 registros
Valor Total: R$ 259,623.56
Valor Médio: R$ 2,472.61


Distribuição por Status:
  MATCHED              -  61 registros (R$ 150,501.32)
  UNMATCHED            -  29 registros (R$ 66,350.59)
  UNMATCHED_MOVEMENT   -  15 registros (R$ 42,771.65)

Top 5 Suppliers:
  1861                           -  25 registros (R$ 68,497.53)
  NULL                           -  15 registros (R$ 42,771.65)
  421                            -  13 registros (R$ 25,876.80)
  1720                           -  13 registros (R$ 26,830.85)
  2383                           -  10 registros (R$ 29,296.14)



In [29]:
# Criar DataFrame e salvar em CSV
import subprocess

script = """
import duckdb
import pandas as pd

# Conectar ao banco
conn = duckdb.connect('/home/Projetos/ProjetosDocker/data/analysis.duckdb', read_only=True)

# Query para criar o DataFrame (R$ 1.000 - R$ 5.000)
df = conn.execute('''
    SELECT * 
    FROM conciliacoes 
    WHERE amount BETWEEN 1000 AND 5000
    ORDER BY amount DESC
''').df()

print(f'✓ DataFrame criado com {len(df)} registros')
print(f'✓ Colunas: {len(df.columns)}')
print(f'\\nPrimeiras 5 linhas:')
print(df[['id', 'status', 'source', 'amount', 'supplier']].head())

# Salvar em CSV
csv_path = '/home/Projetos/ProjetosDocker/data/df_1000_5000.csv'
df.to_csv(csv_path, index=False)
print(f'\\n✓ DataFrame salvo em: df_1000_5000.csv')

# Estatísticas
print(f'\\nEstatísticas do DataFrame:')
print(df[['amount', 'match_score']].describe())

conn.close()
"""

cmd_base = '/home/Projetos/ProjetosDocker/data/.venv/bin/python'
result = subprocess.run([cmd_base, '-c', script], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('Erros:', result.stderr)

✓ DataFrame criado com 105 registros
✓ Colunas: 19

Primeiras 5 linhas:
                                     id  ...                        supplier
0  1e43a283-161a-4a68-b82f-77680783ad83  ...                            1720
1  74611431-29f7-43c4-a161-0eba8d4522d4  ...  Ademar Freire de Carvalho Neto
2  38ac6c36-fef3-4b3f-beed-4e30035a4f43  ...                            1861
3  b5086840-ce03-424b-b801-eeeef46d345e  ...                            1861
4  22e47efe-6480-421b-8198-2f96fc0a6c9e  ...       Receita Federal do Brasil

[5 rows x 5 columns]

✓ DataFrame salvo em: df_1000_5000.csv

Estatísticas do DataFrame:
            amount  match_score
count   105.000000   105.000000
mean   2472.605333     0.657143
std    1213.024003     2.027828
min    1000.000000     0.000000
25%    1429.610000     0.000000
50%    2227.890000     0.000000
75%    3485.000000     0.000000
max    5000.000000    10.000000



In [31]:
# Acessar e manipular o DataFrame salvo
import subprocess

print('📊 Operações com o DataFrame criado\n')

# 1. Ler o CSV criado
script = """
import pandas as pd

df = pd.read_csv('/home/Projetos/ProjetosDocker/data/df_1000_5000.csv')

print('1️⃣  Agrupar por Status:')
print(df.groupby('status')['amount'].agg(['count', 'sum', 'mean']))

print('\\n2️⃣  Agrupar por Source:')
print(df.groupby('source')['amount'].agg(['count', 'sum']))

print('\\n3️⃣  Top 5 por Valor:')
print(df.nlargest(5, 'amount')[['status', 'amount', 'supplier', 'match_reason']])

print('\\n4️⃣  Registros UNMATCHED:')
unmatched = df[df['status'] == 'UNMATCHED']
print(f'Total UNMATCHED: {len(unmatched)} registros (R$ {unmatched["amount"].sum():,.2f})')
"""

cmd_base = '/home/Projetos/ProjetosDocker/data/.venv/bin/python'
result = subprocess.run([cmd_base, '-c', script], capture_output=True, text=True)
print(result.stdout)

📊 Operações com o DataFrame criado

1️⃣  Agrupar por Status:
                    count        sum         mean
status                                           
MATCHED                61  150501.32  2467.234754
UNMATCHED              29   66350.59  2287.951379
UNMATCHED_MOVEMENT     15   42771.65  2851.443333

2️⃣  Agrupar por Source:
          count        sum
source                    
MOVEMENT     15   42771.65
OPEN         29   66350.59
PAID         61  150501.32

3️⃣  Top 5 por Valor:
      status   amount                        supplier       match_reason
0    MATCHED  5000.00                            1720       VALUE_UNIQUE
1  UNMATCHED  4981.00  Ademar Freire de Carvalho Neto   PENDING_NO_MATCH
2    MATCHED  4921.46                            1861  PAID_NO_STATEMENT
3    MATCHED  4841.64                            1861  PAID_NO_STATEMENT
4  UNMATCHED  4756.84       Receita Federal do Brasil   PENDING_NO_MATCH

4️⃣  Registros UNMATCHED:
Total UNMATCHED: 29 registros (R$ 66,350

In [32]:
# Resumo de todos os DataFrames/CSVs disponíveis
import os
import glob

print('📂 DataFrames/CSVs disponíveis no projeto:\n')
print('='*70)

csv_files = sorted(glob.glob('/home/Projetos/ProjetosDocker/data/*.csv'))

for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    size = os.path.getsize(csv_file) / 1024  # KB
    
    # Origem dos dados
    if filename.startswith('data-'):
        tipo = '🔴 CSV Original'
    elif filename.startswith('df_'):
        tipo = '🟢 DataFrame Exportado'
    else:
        tipo = '🔵 Outro'
    
    print(f'{tipo:25s} | {filename:35s} | {size:8.1f} KB')

print('='*70)
print(f'\n💡 Para usar qualquer CSV como DataFrame:')
print(f"   import pandas as pd")
print(f"   df = pd.read_csv('/home/Projetos/ProjetosDocker/data/df_1000_5000.csv')")
print(f'\n🗄️  Banco DuckDB: analysis.duckdb (tabela: conciliacoes)')

📂 DataFrames/CSVs disponíveis no projeto:

🔴 CSV Original            | data-1772046577622.csv              |    135.2 KB
🔴 CSV Original            | data-1772046611038.csv              |      1.1 KB
🔴 CSV Original            | data-1772046636697.csv              |    144.9 KB
🔴 CSV Original            | data-1772046649147.csv              |     55.9 KB
🔴 CSV Original            | data-1772046923950.csv              |      2.8 KB
🟢 DataFrame Exportado     | df_1000_5000.csv                    |     31.0 KB
🟢 DataFrame Exportado     | df_conciliacoes_entradas.csv        |    114.6 KB
🟢 DataFrame Exportado     | df_matched.csv                      |     81.8 KB
🟢 DataFrame Exportado     | df_unmatched.csv                    |     33.1 KB

💡 Para usar qualquer CSV como DataFrame:
   import pandas as pd
   df = pd.read_csv('/home/Projetos/ProjetosDocker/data/df_1000_5000.csv')

🗄️  Banco DuckDB: analysis.duckdb (tabela: conciliacoes)
